## Maximal Marginal Relevance
```"How can we pick results that are not only relevant to the query but different from each other?"```

- **MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query**

In [1]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17432\1530976060.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
d:\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# sample docs
docs = [
    Document(page_content="LangChain is a framework for developing applications powered by language models."),
    Document(page_content="LangChain is used to build applications that integrate with language models."),
    Document(page_content="ChromaDB is a vector database that can be used to store and retrieve embeddings."),
    Document(page_content="Embeddings are numerical representations of text that can be used for similarity search and other tasks."),
    Document(page_content="LangChain provides tools for working with embeddings and vector databases."),
]

In [3]:
embeddings_model = HuggingFaceEndpointEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2",
)

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings_model,
    collection_name='My-Collection'
)

In [16]:
retriver = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 3 , "lambda_mult":0}
)

In [18]:
query = "What is LangChain?"
result = retriver.invoke(query)

In [19]:
for i,doc in enumerate(result):
    print(f"Result {i+1} : ","\n",doc)

Result 1 :  
 page_content='LangChain is a framework for developing applications powered by language models.'
Result 2 :  
 page_content='ChromaDB is a vector database that can be used to store and retrieve embeddings.'
Result 3 :  
 page_content='Embeddings are numerical representations of text that can be used for similarity search and other tasks.'


## Multi-Query Retriever

- Takes your original query
- Uses LLM to generate multiple semantically different versions of that query
- Performs retrieval for each-sub query
- Combines and de-duplicates the results

``` User ki query ko improve krta hai aur uske query se multiple query generate krta h and then retrieve for all the sub queries```


## Contextual Compression Retriever

Improves retrieval quality by commpressing documnets after retrieval - keeping only the relevant content based on the user's query

* How it works
1. Base Retriever (eg FAISS,Chroma) retrieves N documents
2. A compressor (usually LLM) is applied to each document
3. Compressor keeps only the parts relevant to query
4. Irrelevant content is discarded